# Пайплайны (Pipelines)

In [3]:
# загрузим основные библиотеки
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score

Попробуем предсказать цену на недвижимость в Калифорнии

### California Housing Dataset

* MedInc — медианный уровень дохода в квартале;
* HouseAge — медианный возраст дома в квартале;
* AveRooms — среднее количество помещений;
* AveBedrms — среднее количество спальных комнат;
* Population — население квартала;
* AveOccup — средний срок проживания;
* Latitude — значение широты квартала;
* Longitude — значение долготы квартала;
* Price — целевое значение.

## Часть 1. Простейшие пайплайны


Загрузим данные

In [4]:
data = fetch_california_housing()

In [5]:
df = pd.DataFrame(data['data'], columns=data['feature_names'])
df.loc[:, 'target'] = data['target']
df.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


In [6]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [7]:
print(f'Размер обучающей выборки {X_train.shape}')
print(f'Размер тестовой выборки {X_test.shape}')

Размер обучающей выборки (15480, 8)
Размер тестовой выборки (5160, 8)


In [8]:
pipeline = Pipeline([('scaler', StandardScaler()), ('rf', RandomForestRegressor())])
pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2


In [9]:
y_pred = pipeline.predict(X_test)
print(f'Качество по метрике R2: {round(r2_score(y_test, y_pred), 4)}')
print(f'Качество по RSME: {round(root_mean_squared_error(y_test, y_pred), 4)}')

Качество по метрике R2: 0.8081
Качество по RSME: 0.5039


In [10]:
pipeline.get_params()

{'memory': None,
 'steps': [('scaler', StandardScaler()), ('rf', RandomForestRegressor())],
 'transform_input': None,
 'verbose': False,
 'scaler': StandardScaler(),
 'rf': RandomForestRegressor(),
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True,
 'rf__bootstrap': True,
 'rf__ccp_alpha': 0.0,
 'rf__criterion': 'squared_error',
 'rf__max_depth': None,
 'rf__max_features': 1.0,
 'rf__max_leaf_nodes': None,
 'rf__max_samples': None,
 'rf__min_impurity_decrease': 0.0,
 'rf__min_samples_leaf': 1,
 'rf__min_samples_split': 2,
 'rf__min_weight_fraction_leaf': 0.0,
 'rf__monotonic_cst': None,
 'rf__n_estimators': 100,
 'rf__n_jobs': None,
 'rf__oob_score': False,
 'rf__random_state': None,
 'rf__verbose': 0,
 'rf__warm_start': False}

In [11]:
print(pipeline[1].n_estimators) # либо по ключу, либо по названию
print(pipeline['rf'].n_estimators)

100
100


In [12]:
pipeline.set_params(rf__n_estimators=200) # можно поменять параметры

,steps,"[('scaler', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2


In [13]:
from sklearn.model_selection import GridSearchCV

param_grid = {'scaler__with_mean':[True, False],
              'rf__n_estimators':[100, 200, 500]}

grid_search = GridSearchCV(
    pipeline, 
    param_grid=param_grid, 
    verbose = True
)

In [14]:
grid_search.fit(X_train, y_train)
print(grid_search.best_estimator_)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Pipeline(steps=[('scaler', StandardScaler()),
                ('rf', RandomForestRegressor(n_estimators=500))])


In [15]:
y_pred = grid_search.best_estimator_.predict(X_test)

print(f'Качество по метрике R2: { round(r2_score(y_test, y_pred), 4)}')
print(f'Качество по RSME: {round(root_mean_squared_error(y_test, y_pred), 4)}')

Качество по метрике R2: 0.81
Качество по RSME: 0.5015


# Часть 2. Предобработка в пайплайнах

При использовании пайплайна для сборки обработки данных в один стек удобно воспользоваться **Column Transformer**.

**Column Transformer** — это специальный объект из модуля compose библиотеки sklearn, который позволяет применять набор трансформаций к данным. Этот объект позволяет преобразовывать разные столбцы или подмножества столбцов входных данных по отдельности, а результаты, сгенерированные каждым преобразователем, будут объединены в единую таблицу.

Однако, самим Column Transformer пользоваться не совсем удобно, для более удобной работы с ним мы будем использовать «обёртку» в виде функции **make_column_transformer()** из того же модуля библиотеки sklearn.

In [16]:
df_wine = pd.read_csv('Data/Red.csv')

In [17]:
df_wine.head()

,Name,Country,Region,Winery,Rating,NumberOfRatings,Price,Year
0,Pomerol 2011,France,Pomerol,Château La Providence,4.2,100,95.00,2011
1,Lirac 2017,France,Lirac,Château Mont-Redon,4.3,100,15.50,2017
2,Erta e China Rosso di Toscana 2015,Italy,Toscana,Renzo Masi,3.9,100,7.45,2015
3,Bardolino 2019,Italy,Bardolino,Cavalchina,3.5,100,8.72,2019
4,Ried Scheibner Pinot Noir 2016,Austria,Carnuntum,Markowitsch,3.9,100,29.15,2016


In [18]:
df_wine.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8666 entries, 0 to 8665
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             8666 non-null   object 
 1   Country          8666 non-null   object 
 2   Region           8666 non-null   object 
 3   Winery           8666 non-null   object 
 4   Rating           8666 non-null   float64
 5   NumberOfRatings  8666 non-null   int64  
 6   Price            8666 non-null   float64
 7   Year             8666 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 541.8+ KB


Примечание: Если мы хотим применить, например, OneHotEncoder, к более чем одному признаку, то просто достаточно добавить в список колонки, например ['Country', ‘Region’]. Теперь OneHotEncoder будет работать не только на признаке Country, но ещё и на Region.

In [19]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import make_column_transformer

ct = make_column_transformer(
    (OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ['Region']),
    (StandardScaler(), ['Price']), # применяем отдельный скейлер к признаку
    (OneHotEncoder(handle_unknown='ignore'), ['Country']))
print(ct)

ColumnTransformer(transformers=[('ordinalencoder',
                                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['Region']),
                                ('standardscaler', StandardScaler(), ['Price']),
                                ('onehotencoder',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['Country'])])


In [20]:
pipeline = Pipeline([
    ('ct', ct), 
    ('rf', RandomForestRegressor(random_state=42))])

In [21]:
X = df_wine[['Region', 'Price', 'Country']]
y = df_wine['Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [22]:
pipeline.fit(X, y)

,steps,"[('ct', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('ordinalencoder', ...), ('standardscaler', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [23]:
# Получаем трансформированный массив
X_transformed = pipeline['ct'].transform(X_train).toarray()

# Собираем имена всех признаков
region_cols = ['Region']  # OrdinalEncoder не даёт имен
price_cols = ['Price']
country_cols = pipeline['ct'].transformers_[2][1].get_feature_names_out(['Country'])

all_cols = region_cols + price_cols + country_cols.tolist()

# Преобразуем в DataFrame
df_transformed = pd.DataFrame(X_transformed, columns=all_cols)

df_transformed.head()

,Region,Price,Country_Argentina,Country_Australia,Country_Austria,Country_Brazil,Country_Bulgaria,Country_Canada,Country_Chile,Country_China,...,Country_Portugal,Country_Romania,Country_Slovakia,Country_Slovenia,Country_South Africa,Country_Spain,Country_Switzerland,Country_Turkey,Country_United States,Country_Uruguay
0,497.0,-0.387313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,433.0,-0.167724,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,434.0,-0.370241,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,31.0,-0.360939,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,531.0,-0.320200,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
df_wine_test = pd.read_csv('Data/Red_test.csv')
X_test = df_wine_test[['Country', 'Region', 'Price']]
y_test = df_wine_test['Rating']

y_pred = pipeline.predict(X_test)
print(f'Качество по RSME: {round(root_mean_squared_error(y_test, y_pred), 4)}')

Качество по RSME: 0.0765


### Сериализация

In [25]:
# !pip install joblib
import joblib

joblib.dump(pipeline, 'Data/pipeline.pkl')

['Data/pipeline.pkl']

### Десериализация

In [26]:
pipeline_loaded = joblib.load('Data/pipeline.pkl')

In [30]:
print(pipeline_loaded)

Pipeline(steps=[('ct',
                 ColumnTransformer(transformers=[('ordinalencoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['Region']),
                                                 ('standardscaler',
                                                  StandardScaler(), ['Price']),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country'])])),
                ('rf', RandomForestRegressor(random_state=42))])


Теперь попробуем изменить параметры случайного леса в пайплайне, полученном в предыдущем задании.

Измените параметр n_estimators в случайном лесу со значения по умолчанию до 200 , используя метод set_params.

В качестве ответа на задание введите в поле ниже полученный результат по метрике RMSE, округленный до четвёртого знака после запятой.

In [31]:
pipeline_loaded.get_params()

{'memory': None,
 'steps': [('ct',
   ColumnTransformer(transformers=[('ordinalencoder',
                                    OrdinalEncoder(handle_unknown='use_encoded_value',
                                                   unknown_value=-1),
                                    ['Region']),
                                   ('standardscaler', StandardScaler(), ['Price']),
                                   ('onehotencoder',
                                    OneHotEncoder(handle_unknown='ignore'),
                                    ['Country'])])),
  ('rf', RandomForestRegressor(random_state=42))],
 'transform_input': None,
 'verbose': False,
 'ct': ColumnTransformer(transformers=[('ordinalencoder',
                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                 unknown_value=-1),
                                  ['Region']),
                                 ('standardscaler', StandardScaler(), ['Price']),
   

In [32]:
pipeline_loaded.set_params(rf__n_estimators=200)

X = df_wine[['Region', 'Price', 'Country']]
y = df_wine['Rating']

pipeline.fit(X, y)

df_wine_test = pd.read_csv('Data/Red_test.csv')
X_test = df_wine_test[['Country', 'Region', 'Price']]
y_test = df_wine_test['Rating']

y_pred = pipeline.predict(X_test)
print(f'Качество по RSME: {round(root_mean_squared_error(y_test, y_pred), 4)}')

Качество по RSME: 0.0761


Теперь попробуем добавить стекинг в качестве модели в пайплайн.

Нам следует выполнить следующее:

1. Собрать StackingRegressor:
2. В качестве базовых моделей возьмите ридж-регрессию RidgeCV() и решающее дерево.
3. В качестве метамодели возьмите случайный лес с настройками (количество базовых моделей 10).
4. Все базовые модели стекинга модели должны быть с настройками по умолчанию (кроме random_state).
5. Зафиксировать random_state=42 (для всех моделей).
6. Заменить в пайплайне задачи 6.1 случайный лес на StackingRegressor.
7. Обучить модель на тренировочной выборке.
7. В качестве ответа на задание ввести в поле ниже полученный результат по метрике RMSE, округлённый до второго знака после запятой.

In [33]:
df_wine= pd.read_csv('Data/Red.csv')
X = df_wine[['Country', 'Price', 'Region']]
y = df_wine['Rating']

# Cписок базовых моделей
base_estimators = [
    ('ridge', RidgeCV()),
    ('tree', DecisionTreeRegressor(random_state=42))
]

stack = StackingRegressor(
        estimators=base_estimators,
        final_estimator=RandomForestRegressor(n_estimators=10, random_state=42), 
        n_jobs=-1)

ct = make_column_transformer(
    (OrdinalEncoder(), ['Region']),
     (StandardScaler(), ['Price']),
    (OneHotEncoder(), ['Country']),
)

# Пайплайн со стекингом
pipeline_stacking = Pipeline([
    ('ct', ct),
    ('stack', stack)
    ])

pipeline_stacking.fit(X, y)

y_pred = pipeline_stacking.predict(X_test)
print(f'Качество по RSME: {round(root_mean_squared_error(y_test, y_pred), 2)}')

/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/pyth

Качество по RSME: 0.18
